# 08 — Speaker Recognition & Verification

Dans ce notebook, nous allons apprendre à représenter une voix sous forme de speaker embedding, puis à comparer plusieurs voix.

Pipeline principal :

```text
Audio
→ Speaker Encoder
→ Speaker Embedding
→ Cosine Similarity
→ Threshold
→ Same Speaker / Different Speaker

Objectifs: 

```text
charger un modèle préentraîné de speaker recognition ;
transformer un audio en speaker embedding ;
comparer deux embeddings avec la cosine similarity ;
comprendre le principe d'enrollment ;
effectuer une vérification 1:1 ;
tester plusieurs extraits d'un même speaker ;
comparer avec un speaker différent ;
comprendre le rôle du threshold ;
introduire les notions de False Accept, False Reject et EER.

In [ ]:
!python -m pip install -U speechbrain

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
import torch.nn.functional as F

In [ ]:
import speechbrain

print("SpeechBrain version :", speechbrain.__version__)

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier

speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="outputs/spkrec-ecapa-voxceleb"
)

print("ECAPA-TDNN chargé.")

In [ ]:
from pathlib import Path
import torch
import torchaudio

audio_path = Path("audio_samples/voice_excerpt.wav")

waveform, sr = torchaudio.load(str(audio_path))

# Conversion mono si nécessaire
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0, keepdim=True)

# Extrait 20s → 30s
start_time = 20
duration = 10

start_sample = int(start_time * sr)
end_sample = int((start_time + duration) * sr)

audio_segment = waveform[:, start_sample:end_sample]

print("Sample rate :", sr)
print("Shape :", audio_segment.shape)
print("Duration :", audio_segment.shape[1] / sr)

In [ ]:
if sr != 16000:
    resampler = torchaudio.transforms.Resample(
        orig_freq=sr,
        new_freq=16000
    )
    audio_16k = resampler(audio_segment)
else:
    audio_16k = audio_segment

print("Waveform 16 kHz :", audio_16k.shape)

In [ ]:
with torch.no_grad():
    embedding_1 = speaker_model.encode_batch(audio_16k)

print("Embedding shape :", embedding_1.shape)

In [ ]:
start_time_2 = 40
duration_2 = 10

start_sample_2 = int(start_time_2 * sr)
end_sample_2 = int((start_time_2 + duration_2) * sr)

audio_segment_2 = waveform[:, start_sample_2:end_sample_2]

if sr != 16000:
    audio_16k_2 = resampler(audio_segment_2)
else:
    audio_16k_2 = audio_segment_2

with torch.no_grad():
    embedding_2 = speaker_model.encode_batch(audio_16k_2)

print("Embedding 2 shape :", embedding_2.shape)

In [ ]:
import torch.nn.functional as F

similarity = F.cosine_similarity(
    embedding_1.squeeze(),
    embedding_2.squeeze(),
    dim=0
)

print("Cosine similarity :", similarity.item())

In [ ]:
!python -m pip install -U datasets

In [ ]:
!python -m pip install -U torchcodec

In [ ]:
from datasets import load_dataset, Audio

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation"
)

dataset = dataset.cast_column(
    "audio",
    Audio(decode=False)
)

example = dataset[0]

print("Speaker ID :", example["speaker_id"])
print("Audio :", example["audio"])

In [ ]:
from pathlib import Path

example = dataset[0]

speaker_2_path = Path("audio_samples/speaker_1272.flac")

with open(speaker_2_path, "wb") as f:
    f.write(example["audio"]["bytes"])

print("Audio sauvegardé :", speaker_2_path)
print("Speaker ID :", example["speaker_id"])

In [ ]:
import torch
import torchaudio

speaker_2_waveform, speaker_2_sr = torchaudio.load(
    "audio_samples/speaker_1272.flac"
)

if speaker_2_waveform.shape[0] > 1:
    speaker_2_waveform = speaker_2_waveform.mean(dim=0, keepdim=True)

if speaker_2_sr != 16000:
    resampler_2 = torchaudio.transforms.Resample(
        orig_freq=speaker_2_sr,
        new_freq=16000
    )
    speaker_2_16k = resampler_2(speaker_2_waveform)
else:
    speaker_2_16k = speaker_2_waveform

print("Sample rate :", speaker_2_sr)
print("Waveform shape :", speaker_2_16k.shape)

In [ ]:
with torch.no_grad():
    embedding_3 = speaker_model.encode_batch(speaker_2_16k)

print("Embedding shape :", embedding_3.shape)

In [ ]:
import torch.nn.functional as F

similarity_different = F.cosine_similarity(
    embedding_1.squeeze(),
    embedding_3.squeeze(),
    dim=0
)

print(
    "Cosine similarity - different speakers :",
    similarity_different.item()
)

## Speaker Verification Results

Nous avons comparé les embeddings ECAPA-TDNN avec la cosine similarity.

| Comparison | Cosine Similarity |
|---|---:|
| Same speaker | 0.778 |
| Different speakers | -0.044 |

Le modèle produit une similarité nettement plus élevée pour deux extraits provenant du même speaker.

Cependant, ces deux exemples ne suffisent pas pour définir un threshold fiable.  
Pour cela, il faut comparer plusieurs paires `same speaker` et `different speaker`, puis étudier les erreurs de type False Accept et False Reject.

In [ ]:
same_scores = [0.778]
different_scores = [-0.044]

threshold = 0.5

def verify_speaker(score, threshold=0.5):
    return "SAME SPEAKER" if score >= threshold else "DIFFERENT SPEAKER"

print("Same speaker test :", verify_speaker(same_scores[0], threshold))
print("Different speaker test :", verify_speaker(different_scores[0], threshold))

## Threshold

Le système transforme la cosine similarity en décision :

```text
score >= threshold
→ SAME SPEAKER

score < threshold
→ DIFFERENT SPEAKER

In [ ]:
same_scores = [0.82, 0.76, 0.71, 0.64, 0.55, 0.48]
different_scores = [0.32, 0.21, 0.08, -0.04, 0.44, 0.52]

threshold = 0.5

false_rejects = sum(score < threshold for score in same_scores)
false_accepts = sum(score >= threshold for score in different_scores)

FRR = false_rejects / len(same_scores)
FAR = false_accepts / len(different_scores)

print("False Rejects :", false_rejects)
print("False Accepts :", false_accepts)
print("FRR :", FRR)
print("FAR :", FAR)

## FAR, FRR and EER

Deux erreurs sont importantes en speaker verification :

- **False Accept Rate (FAR)** : le système accepte un mauvais speaker.
- **False Reject Rate (FRR)** : le système refuse le bon speaker.

Le threshold contrôle l'équilibre entre les deux.

Si le threshold est trop faible :
→ plus de False Accepts.

Si le threshold est trop élevé :
→ plus de False Rejects.

Le **Equal Error Rate (EER)** correspond approximativement au point où :

```text
FAR ≈ FRR

In [ ]:
import numpy as np

thresholds = np.linspace(-0.1, 0.9, 101)

results = []

for threshold in thresholds:
    false_rejects = sum(score < threshold for score in same_scores)
    false_accepts = sum(score >= threshold for score in different_scores)

    FRR = false_rejects / len(same_scores)
    FAR = false_accepts / len(different_scores)

    results.append((threshold, FAR, FRR))

eer_threshold, eer_far, eer_frr = min(
    results,
    key=lambda x: abs(x[1] - x[2])
)

eer = (eer_far + eer_frr) / 2

print("Approximate EER threshold :", eer_threshold)
print("FAR :", eer_far)
print("FRR :", eer_frr)
print("Approximate EER :", eer)

## Approximate EER

Nous testons plusieurs thresholds et calculons pour chacun :

- FAR
- FRR

Nous cherchons ensuite le threshold où :

```text
FAR ≈ FRR

In [ ]:
import matplotlib.pyplot as plt

threshold_values = [r[0] for r in results]
far_values = [r[1] for r in results]
frr_values = [r[2] for r in results]

plt.figure(figsize=(8, 5))

plt.plot(threshold_values, far_values, label="FAR")
plt.plot(threshold_values, frr_values, label="FRR")

plt.axvline(
    eer_threshold,
    linestyle="--",
    label=f"EER threshold ≈ {eer_threshold:.2f}"
)

plt.xlabel("Threshold")
plt.ylabel("Error Rate")
plt.title("FAR and FRR vs Threshold")
plt.legend()
plt.grid(True)

plt.show()

## Conclusion

Dans ce notebook, nous avons étudié le principe de speaker recognition et de speaker verification avec ECAPA-TDNN.

Nous avons appris à :

- charger un modèle préentraîné avec SpeechBrain ;
- transformer un extrait audio en speaker embedding ;
- comparer deux embeddings avec la cosine similarity ;
- vérifier deux extraits du même speaker ;
- comparer avec un speaker différent ;
- utiliser un threshold pour prendre une décision ;
- comprendre les notions de False Accept et False Reject ;
- calculer FAR et FRR ;
- estimer approximativement l'EER ;
- visualiser l'effet du threshold sur FAR et FRR.

### Résultats principaux

- Same speaker cosine similarity : **0.778**
- Different speakers cosine similarity : **-0.044**

Le modèle sépare donc clairement les deux speakers dans notre expérience.

Le threshold utilisé dans ce notebook reste illustratif.  
Dans un vrai système, il doit être calibré sur beaucoup plus de paires de speakers et évalué sur un ensemble de validation ou de test.